In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpile วงจรจากระยะไกลด้วย Qiskit Transpiler Service

> **Danger:** ตั้งแต่วันที่ 18 กรกฎาคม 2025 บริการนี้กำลังถูกย้ายเพื่อรองรับ IBM Quantum&reg; Platform ใหม่และยังไม่พร้อมให้ใช้งาน สำหรับ AI passes สามารถใช้ [โหมด local](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud) ได้
> 
>     บริการนี้เป็น beta release ซึ่งอาจมีการเปลี่ยนแปลงได้
>     หากมีความคิดเห็นหรือต้องการติดต่อทีมพัฒนา ใช้ช่องทาง [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) นี้

Qiskit Transpiler Service ให้ความสามารถในการ transpile บนคลาวด์ นอกจากความสามารถของ Qiskit transpiler แบบ local แล้ว งาน transpilation ของคุณยังสามารถใช้ประโยชน์จากทั้งทรัพยากรคลาวด์ IBM Quantum และ transpiler passes ที่ขับเคลื่อนด้วย AI

Qiskit Transpiler Service มีไลบรารี Python เพื่อผสานรวมบริการและความสามารถนี้เข้ากับ Qiskit patterns และ workflows ที่มีอยู่ได้อย่างราบรื่น บริการนี้ใช้ได้เฉพาะผู้ใช้ IBM Quantum Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น

<span id="install-transpiler-service"></span>
## ติดตั้งแพ็กเกจ qiskit-ibm-transpiler
หากต้องการใช้ Qiskit Transpiler Service ให้ติดตั้งแพ็กเกจ `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

แพ็กเกจจะตรวจสอบสิทธิ์โดยอัตโนมัติโดยใช้ [ข้อมูลรับรอง IBM Quantum Platform](/guides/cloud-setup) ของคุณ ซึ่งสอดคล้องกับวิธีที่ [Qiskit Runtime จัดการ](/guides/initialize-account):
- ตัวแปรสภาพแวดล้อม: `QISKIT_IBM_TOKEN`
- ไฟล์ configuration `~/.qiskit/qiskit-ibm.json` (ในส่วน `default-ibm-quantum`)

*หมายเหตุ*: แพ็กเกจนี้ต้องใช้ Qiskit SDK v1.X

## ตัวเลือก transpile ของ qiskit-ibm-transpiler
- `backend_name` (ไม่บังคับ, str) - ชื่อ Backend ตามที่ QiskitRuntimeService คาดหวัง (เช่น `ibm_torino`) หากตั้งค่านี้ไว้ เมธอด transpile จะใช้ layout จาก Backend ที่ระบุสำหรับการ transpile หากมีตัวเลือกอื่นที่ส่งผลต่อการตั้งค่าเหล่านี้ เช่น `coupling_map` การตั้งค่า `backend_name` จะถูกแทนที่
- `coupling_map` (ไม่บังคับ, List[List[int]]) - รายการ coupling map ที่ถูกต้อง (เช่น [[0,1],[1,2]]) หากตั้งค่านี้ไว้ เมธอด transpile จะใช้ coupling map นี้สำหรับการ transpile หากกำหนดไว้จะแทนที่ค่าที่ระบุสำหรับ `target`
- `optimization_level` (int) - ระดับการ optimize ที่จะใช้ในกระบวนการ transpile ค่าที่ถูกต้องคือ [1,2,3] โดย 1 คือการ optimize น้อยที่สุด (และเร็วที่สุด) และ 3 คือการ optimize มากที่สุด (และใช้เวลามากที่สุด)
- `ai` ("true", "false", "auto") - ว่าจะใช้ความสามารถที่ขับเคลื่อนด้วย AI ระหว่างการ transpile หรือไม่ ความสามารถที่ขับเคลื่อนด้วย AI ที่มีให้ใช้งานได้แก่ `AIRouting` transpiling passes หรือวิธีสังเคราะห์ที่ขับเคลื่อนด้วย AI อื่นๆ หากค่านี้เป็น `"true"` บริการจะใช้ transpiling passes ที่ขับเคลื่อนด้วย AI ต่างกันขึ้นอยู่กับ `optimization_level` ที่ร้องขอ หาก `"false"` จะใช้คุณสมบัติ transpile ล่าสุดของ Qiskit โดยไม่มี AI สุดท้าย หาก `"auto"` บริการจะตัดสินใจว่าจะใช้ Qiskit heuristic passes มาตรฐานหรือ AI-powered passes ตาม Circuit ของคุณ
- `qiskit_transpile_options` (dict) - Python dictionary object ที่สามารถรวมตัวเลือกอื่นๆ ที่ถูกต้องใน [เมธอด `transpile()` ของ Qiskit](defaults-and-configuration-options) หาก `qiskit_transpile_options` มี `optimization_level` จะถูกละทิ้งแทนที่ด้วย `optimization_level` ที่ระบุเป็นพารามิเตอร์ input หาก `qiskit_transpile_options` มีตัวเลือกที่ไม่รู้จักโดยเมธอด `transpile()` ของ Qiskit ไลบรารีจะแสดงข้อผิดพลาด

สำหรับข้อมูลเพิ่มเติมเกี่ยวกับเมธอด `qiskit-ibm-transpiler` ที่มีให้ใช้งาน ดู [qiskit-ibm-transpiler API reference](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) หากต้องการเรียนรู้เพิ่มเติมเกี่ยวกับ service API ดู [Qiskit Transpiler Service REST API documentation](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## ตัวอย่าง
ตัวอย่างต่อไปนี้แสดงวิธี transpile วงจรโดยใช้ Qiskit Transpiler Service ด้วยพารามิเตอร์ต่างๆ

1. สร้าง Circuit และเรียก Qiskit Transpiler Service เพื่อ transpile Circuit โดยใช้ `ibm_torino` เป็น `backend_name`, 3 เป็น `optimization_level` และไม่ใช้ AI ระหว่างการ transpile

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*หมายเหตุ*: คุณใช้ได้เฉพาะอุปกรณ์ backend_name ที่คุณเข้าถึงได้ด้วยบัญชี IBM Quantum ของคุณ นอกจาก `backend_name` แล้ว `TranspilerService` ยังรองรับ `coupling_map` เป็นพารามิเตอร์ด้วย

2. สร้าง Circuit ที่คล้ายกันและ transpile โดยขอความสามารถ AI transpiling ด้วยการตั้งค่าแฟล็ก `ai` เป็น `True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. สร้าง Circuit ที่คล้ายกันและ transpile โดยให้บริการตัดสินใจว่าจะใช้ AI-powered transpiling passes หรือไม่